# Calculate earth movers distance per feature between populations

Inspired by the work done in the [SPACe paper](https://www.nature.com/articles/s41467-024-54264-4#Sec10), we are calculating the earth mover's distance with a sign based on if the median value of a feature is higher or lower than the reference population. The populations we are comparing are:

1. Failing + DMSO (reference) to Failing + drug_x
2. Healthy + DMSO (reference) to Failing + drug_x
3. Failing + DMSO (reference) to Healthy + DMSO

We are using SciPy's implementation of calculating earth mover's distance also known as Wasserstein distance.

In [1]:
import pathlib
import pandas as pd

from scipy.stats import wasserstein_distance
import numpy as np

import sys

sys.path.append("../../../utils")

from emd_utils import compute_signed_emd_per_feature, compute_median_baseline_emd

## Set output directory for calculated EMD per comparison

In [2]:
output_dir = pathlib.Path("./emd_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Output file for EMD thresholds
output_emd_thresholds = pathlib.Path(f"{output_dir}/emd_thresholds.csv")
# Define the columns you want
columns = ["Reference", "Comparison", "EMD_threshold_value"]
# Create an empty DataFrame with just the header
pd.DataFrame(columns=columns).to_csv(output_emd_thresholds, index=False)

## Load in data with drug_x and controls

In [3]:
# Directory containing the data files
data_dir = pathlib.Path("../../../3.process_cfret_features/data/single_cell_profiles")

# Load in "plate 3" data that contains the control and drug_x conditions
data_df = pd.read_parquet(
    pathlib.Path(data_dir, "localhost230405150001_sc_normalized.parquet")
)

# Drop columns that don't start with Metadata_ and contain 'Location', 'Parent', or 'Child'
cols_to_drop = [
    col
    for col in data_df.columns
    if not col.startswith("Metadata_")
    and (
        "Location" in col
        or "Parent" in col
        or "Child" in col
        or "Number" in col
        or "Neighbors" in col
    )  # Drop neighbors due to low feature numbers
]
data_df = data_df.drop(columns=cols_to_drop)

# Print dataframe
print(data_df.shape)
data_df.head()

(20865, 1906)


,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,...,Nuclei_Texture_Variance_Hoechst_3_02_256,Nuclei_Texture_Variance_Hoechst_3_03_256,Nuclei_Texture_Variance_Mitochondria_3_00_256,Nuclei_Texture_Variance_Mitochondria_3_01_256,Nuclei_Texture_Variance_Mitochondria_3_02_256,Nuclei_Texture_Variance_Mitochondria_3_03_256,Nuclei_Texture_Variance_PM_3_00_256,Nuclei_Texture_Variance_PM_3_01_256,Nuclei_Texture_Variance_PM_3_02_256,Nuclei_Texture_Variance_PM_3_03_256
0,B,2,9,failing,rejected,DMSO,221.046761,137.115493,246.602800,109.285755,...,-0.361718,-0.361815,-0.278543,-0.278748,-0.281016,-0.280648,-0.321748,-0.324540,-0.326290,-0.325595
1,B,2,9,failing,rejected,DMSO,690.596142,183.067828,716.170091,177.132195,...,-0.371930,-0.373654,2.297362,2.519445,2.039900,1.946821,-0.204002,-0.190676,-0.215768,-0.227077
2,B,2,9,failing,rejected,DMSO,626.561490,206.923698,623.943740,199.906440,...,-0.408321,-0.408046,-0.254027,-0.252227,-0.255444,-0.256202,-0.331222,-0.330529,-0.332888,-0.334313
3,B,2,9,failing,rejected,DMSO,559.448583,220.688160,528.646623,196.955552,...,-0.372755,-0.370338,-0.168390,-0.168077,-0.168909,-0.163799,-0.239871,-0.229786,-0.230989,-0.240673
4,B,2,9,failing,rejected,DMSO,909.019946,247.694340,897.965996,253.621836,...,-0.402623,-0.401941,0.015661,0.061135,0.059207,0.000078,-0.369169,-0.368101,-0.369411,-0.368442


## Split data into the three populations to compare

In [4]:
# Split the data into each population/condition
healthy_DMSO_df = data_df[
    (data_df["Metadata_cell_type"] == "healthy")
    & (data_df["Metadata_treatment"] == "DMSO")
]
failing_drug_x_df = data_df[
    (data_df["Metadata_cell_type"] == "failing")
    & (data_df["Metadata_treatment"] == "drug_x")
]
failing_DMSO_df = data_df[
    (data_df["Metadata_cell_type"] == "failing")
    & (data_df["Metadata_treatment"] == "DMSO")
]

# Print the shapes of the dataframes
print("Healthy DMSO shape:", healthy_DMSO_df.shape)
print("Failing Drug X shape:", failing_drug_x_df.shape)
print("Failing DMSO shape:", failing_DMSO_df.shape)

Healthy DMSO shape: (1366, 1906)
Failing Drug X shape: (3645, 1906)
Failing DMSO shape: (9153, 1906)


## Compute EMD comparing failing drug_x cells to the healthy DMSO cells (reference)

In [5]:
# Compute the median baseline EMD between healthy DMSO and failing drug X
hvfd_threshold_value = compute_median_baseline_emd(
    reference_df=healthy_DMSO_df, comparison_df=failing_drug_x_df, num_permutations=20
)

# Create one-row dataframe to save threshold value to CSV file
hvfd_row = pd.DataFrame(
    [
        {
            "Reference": "healthy_DMSO",
            "Comparison": "failing_drug_x",
            "EMD_threshold_value": hvfd_threshold_value,
        }
    ]
)

# Append to existing CSV file
print("Writing to:", output_emd_thresholds.resolve())
hvfd_row.to_csv(output_emd_thresholds, mode="a", index=False, header=False)
print("Appended row for comparison: failing_DMSO vs failing_drug_x")


# Print the threshold value
print("EMD threshold value:", hvfd_threshold_value)

Writing to: /media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/4.analyze_data/notebooks/EMD_analysis/emd_results/emd_thresholds.csv
Appended row for comparison: failing_DMSO vs failing_drug_x
EMD threshold value: 0.03270299445657731


In [6]:
# Compute the signed EMD for each feature
healthy_vs_failing_drug_x_emd = compute_signed_emd_per_feature(
    reference_df=healthy_DMSO_df, comparison_df=failing_drug_x_df
)

# Print the results
print(healthy_vs_failing_drug_x_emd.shape)
healthy_vs_failing_drug_x_emd.sort_values("signed_emd", ascending=False).head()

(1887, 2)


,feature,signed_emd
88,Cytoplasm_Correlation_K_ER_Actin,1.444714
717,Cells_Correlation_K_ER_Actin,1.439286
1198,Cells_Texture_SumEntropy_Actin_3_00_256,1.268920
1200,Cells_Texture_SumEntropy_Actin_3_02_256,1.268209
96,Cytoplasm_Correlation_K_Mitochondria_Actin,1.265312


### Split feature column into parts

In [7]:
# Split the 'feature' column into new columns as requested (not as a function)
split_df = healthy_vs_failing_drug_x_emd["feature"].str.split("_", n=4, expand=True)
split_df.columns = ["compartment", "feature_group", "measurement", "organelle", "rest"]
valid_organelles = {"Actin", "Mitochondria", "Hoechst", "ER", "PM"}
split_df["organelle"] = split_df["organelle"].where(
    split_df["organelle"].isin(valid_organelles), "Other"
)
healthy_vs_failing_drug_x_emd = pd.concat(
    [
        healthy_vs_failing_drug_x_emd,
        split_df[["compartment", "feature_group", "measurement", "organelle"]],
    ],
    axis=1,
)

# Save the results to parquet file
output_file = output_dir / "healthy_vs_failing_drug_x_emd.parquet"
healthy_vs_failing_drug_x_emd.to_parquet(output_file, index=False)

# Print the final DataFrame
healthy_vs_failing_drug_x_emd.head()

,feature,signed_emd,compartment,feature_group,measurement,organelle
0,Cytoplasm_AreaShape_Area,0.113879,Cytoplasm,AreaShape,Area,Other
1,Cytoplasm_AreaShape_BoundingBoxArea,0.360786,Cytoplasm,AreaShape,BoundingBoxArea,Other
2,Cytoplasm_AreaShape_BoundingBoxMaximum_X,0.075079,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
3,Cytoplasm_AreaShape_BoundingBoxMaximum_Y,0.123542,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
4,Cytoplasm_AreaShape_BoundingBoxMinimum_X,0.069428,Cytoplasm,AreaShape,BoundingBoxMinimum,Other


## Compute EMD comparing failing drug_x cells to the failing DMSO cells (reference)

In [8]:
# Compute the median baseline EMD between healthy DMSO and failing drug X
fvfd_threshold_value = compute_median_baseline_emd(
    reference_df=failing_DMSO_df, comparison_df=failing_drug_x_df, num_permutations=20
)

# Create one-row dataframe to save threshold value to CSV file
fvfd_row = pd.DataFrame(
    [
        {
            "Reference": "failing_DMSO",
            "Comparison": "failing_drug_x",
            "EMD_threshold_value": fvfd_threshold_value,
        }
    ]
)

# Append without header
print("Writing to:", output_emd_thresholds.resolve())
fvfd_row.to_csv(output_emd_thresholds, mode="a", index=False, header=False)
print("Appended row for comparison: failing_DMSO vs failing_drug_x")

# Print the threshold value
print("EMD threshold value:", fvfd_threshold_value)

Writing to: /media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/4.analyze_data/notebooks/EMD_analysis/emd_results/emd_thresholds.csv
Appended row for comparison: failing_DMSO vs failing_drug_x
EMD threshold value: 0.021483669564844047


In [9]:
# Compute the signed EMD for each feature
failing_vs_failing_drug_x_emd = compute_signed_emd_per_feature(
    reference_df=failing_DMSO_df, comparison_df=failing_drug_x_df
)

# Print the results
print(failing_vs_failing_drug_x_emd.shape)
failing_vs_failing_drug_x_emd.sort_values("signed_emd", ascending=False).head()

(1887, 2)


,feature,signed_emd
1165,Cells_Texture_InverseDifferenceMoment_ER_3_03_256,1.604533
1163,Cells_Texture_InverseDifferenceMoment_ER_3_01_256,1.576880
1162,Cells_Texture_InverseDifferenceMoment_ER_3_00_256,1.576071
1002,Cells_Texture_AngularSecondMoment_ER_3_00_256,1.573541
1164,Cells_Texture_InverseDifferenceMoment_ER_3_02_256,1.563669


### Split feature column into parts

In [10]:
# Split the 'feature' column into new columns as requested (not as a function)
split_df = failing_vs_failing_drug_x_emd["feature"].str.split("_", n=4, expand=True)
split_df.columns = ["compartment", "feature_group", "measurement", "organelle", "rest"]
valid_organelles = {"Actin", "Mitochondria", "Hoechst", "ER", "PM"}
split_df["organelle"] = split_df["organelle"].where(
    split_df["organelle"].isin(valid_organelles), "Other"
)
failing_vs_failing_drug_x_emd = pd.concat(
    [
        failing_vs_failing_drug_x_emd,
        split_df[["compartment", "feature_group", "measurement", "organelle"]],
    ],
    axis=1,
)

# Save the results to parquet file
output_file = output_dir / "failing_vs_failing_drug_x_emd.parquet"
failing_vs_failing_drug_x_emd.to_parquet(output_file, index=False)

# Print the final DataFrame
failing_vs_failing_drug_x_emd.head()

,feature,signed_emd,compartment,feature_group,measurement,organelle
0,Cytoplasm_AreaShape_Area,-0.042331,Cytoplasm,AreaShape,Area,Other
1,Cytoplasm_AreaShape_BoundingBoxArea,0.161836,Cytoplasm,AreaShape,BoundingBoxArea,Other
2,Cytoplasm_AreaShape_BoundingBoxMaximum_X,0.033909,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
3,Cytoplasm_AreaShape_BoundingBoxMaximum_Y,0.047710,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
4,Cytoplasm_AreaShape_BoundingBoxMinimum_X,-0.031397,Cytoplasm,AreaShape,BoundingBoxMinimum,Other


## Compute EMD comparing healthy DMSO cells to the failing DMSO cells (reference)

In [11]:
# Compute the median baseline EMD between healthy DMSO and failing drug X
hvf_threshold_value = compute_median_baseline_emd(
    reference_df=failing_DMSO_df, comparison_df=healthy_DMSO_df, num_permutations=20
)

# Create one-row dataframe to save threshold value to CSV file
hvf_row = pd.DataFrame(
    [
        {
            "Reference": "failing_DMSO",
            "Comparison": "healthy_DMSO",
            "EMD_threshold_value": hvf_threshold_value,
        }
    ]
)

# Append without header
print("Writing to:", output_emd_thresholds.resolve())
hvf_row.to_csv(output_emd_thresholds, mode="a", index=False, header=False)
print("Appended row for comparison: failing_DMSO vs failing_drug_x")

# Print the threshold value
print("EMD threshold value:", hvf_threshold_value)

Writing to: /media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/4.analyze_data/notebooks/EMD_analysis/emd_results/emd_thresholds.csv
Appended row for comparison: failing_DMSO vs failing_drug_x
EMD threshold value: 0.031387434820334736


In [12]:
# Compute the signed EMD for each feature
failing_vs_healthy_DMSO_emd = compute_signed_emd_per_feature(
    reference_df=failing_DMSO_df, comparison_df=healthy_DMSO_df
)

# Print the results
print(failing_vs_healthy_DMSO_emd.shape)
failing_vs_healthy_DMSO_emd.sort_values("signed_emd", ascending=False).head()

(1887, 2)


,feature,signed_emd
65,Cytoplasm_Correlation_Costes_Actin_Hoechst,2.134575
532,Cytoplasm_Texture_InverseDifferenceMoment_Acti...,2.076448
1161,Cells_Texture_InverseDifferenceMoment_Actin_3_...,2.067770
531,Cytoplasm_Texture_InverseDifferenceMoment_Acti...,2.065252
1160,Cells_Texture_InverseDifferenceMoment_Actin_3_...,2.055254


### Split feature column into parts

In [13]:
# Split the 'feature' column into new columns as requested (not as a function)
split_df = failing_vs_healthy_DMSO_emd["feature"].str.split("_", n=4, expand=True)
split_df.columns = ["compartment", "feature_group", "measurement", "organelle", "rest"]
valid_organelles = {"Actin", "Mitochondria", "Hoechst", "ER", "PM"}
split_df["organelle"] = split_df["organelle"].where(
    split_df["organelle"].isin(valid_organelles), "Other"
)
failing_vs_healthy_DMSO_emd = pd.concat(
    [
        failing_vs_healthy_DMSO_emd,
        split_df[["compartment", "feature_group", "measurement", "organelle"]],
    ],
    axis=1,
)

# Save the results to parquet file
output_file = output_dir / "failing_vs_healthy_DMSO_emd.parquet"
failing_vs_healthy_DMSO_emd.to_parquet(output_file, index=False)

# Print the final DataFrame
failing_vs_healthy_DMSO_emd.head()

,feature,signed_emd,compartment,feature_group,measurement,organelle
0,Cytoplasm_AreaShape_Area,-0.151724,Cytoplasm,AreaShape,Area,Other
1,Cytoplasm_AreaShape_BoundingBoxArea,0.426730,Cytoplasm,AreaShape,BoundingBoxArea,Other
2,Cytoplasm_AreaShape_BoundingBoxMaximum_X,-0.048934,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
3,Cytoplasm_AreaShape_BoundingBoxMaximum_Y,-0.078523,Cytoplasm,AreaShape,BoundingBoxMaximum,Other
4,Cytoplasm_AreaShape_BoundingBoxMinimum_X,-0.093615,Cytoplasm,AreaShape,BoundingBoxMinimum,Other
